# W8 · Day 2 — Metadata, PII Scrubbing, Pipeline Wiring

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Day 1 taught parsing and chunking. Day 2 covers what makes the pipeline
production-ready: **metadata** (for filtered retrieval) and **PII scrubbing**
(so sensitive data never reaches the vector store). We wrap by wiring
everything into a mini-pipeline on ONE sample document.

**This is the first appearance of the Security + PII recurring thread** —
returns in W14 (tool permissioning), W18 (prompt injection), W25 (observability
data), W28 (data residency deep-dive).

**Cost per full run:** ~$0.001 (a few embedding calls for the Qdrant demo).

**Notebook flow:**
- Cell 1: Setup + hard assertion that Presidio is installed (yesterday's homework)
- Cell 2: The 9-field metadata schema
- Cell 3: Qdrant filter DSL — build a demo collection, run filtered queries
- Cell 4: Why PII in vector stores is a discoverability problem
- Cells 5-8: PII detection — regex baseline, Presidio NER, anonymization, three-legs pattern
- Cell 9: Data residency — name your stack
- Cell 10: Wire it all together on ONE sample document
- Cell 11: Wrap + hand-off to Track B

In [3]:

import os
from getpass import getpass

# Reuse the same Qdrant Cloud URL from Week 7
os.environ["QDRANT_URL"] = "https://1af9eb0e-770d-400a-8855-b7607c63a696.eu-central-1-0.aws.cloud.qdrant.io"

# Prompt for the key without displaying it
if not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass(
        "Enter your Qdrant API key: "
    )

print("Qdrant settings are ready.")


Qdrant settings are ready.


---

## Cell 1 — Setup + Presidio check

Yesterday's homework: `pip install presidio-analyzer presidio-anonymizer` +
`python -m spacy download en_core_web_sm`.

The assertion below fails **hard** if Presidio isn't installed. That's
intentional — this notebook depends on it, and quietly skipping content is
worse than a loud stop.

In [4]:
%pip install presidio-analyzer presidio-anonymizer spacy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys

!{sys.executable} -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.3 MB/s  0:00:00m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [6]:
import sys
print("Python:", sys.executable)

import spacy
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

spacy.load("en_core_web_sm")

print("spaCy: ready")
print("English model: ready")
print("Presidio Analyzer: ready")
print("Presidio Anonymizer: ready")

Python: /usr/bin/python


/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-11-21 09:12:50.831187: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-21 09:12:51.574575: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-21 09:13:02.512008: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-21 09:13:09.555236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for

spaCy: ready
English model: ready
Presidio Analyzer: ready
Presidio Anonymizer: ready


In [7]:
import os
import sys
from pathlib import Path

# ── Presidio check — HARD ASSERTION ──
try:
    from presidio_analyzer import AnalyzerEngine
    from presidio_anonymizer import AnonymizerEngine
except ImportError as e:
    raise ImportError(
        "\n\n"
        "═══════════════════════════════════════════════════════════════\n"
        "PRESIDIO NOT INSTALLED\n"
        "═══════════════════════════════════════════════════════════════\n"
        "This notebook requires Presidio (yesterday's homework).\n\n"
        "Run:\n"
        "   pip install presidio-analyzer presidio-anonymizer\n"
        "   python -m spacy download en_core_web_sm\n\n"
        "Total download: ~150 MB. Then restart the kernel and re-run.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from e

# ── Other imports ──
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — from W7 Qdrant Cloud setup"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from W7 Qdrant Cloud setup"

# Import Day 1's helper module
sys.path.insert(0, str(Path.cwd()))
from wk08_pipeline import (
    SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX,
    parse_pdf, parse_html, parse_docx,
    chunk_recursive,
    scrub_pii_regex, PII_PATTERNS,
    METADATA_FIELDS,
    check_samples_exist,
)

check_samples_exist()
print("Setup ok.")
print(f"  Presidio: installed")
print(f"  OpenAI key set")
print(f"  Qdrant URL: {os.environ['QDRANT_URL'][:40]}...")
print(f"  Sample docs present")
print(f"  Day 1 helpers imported: parse_*, chunk_recursive, scrub_pii_regex, METADATA_FIELDS")

Setup ok.
  Presidio: installed
  OpenAI key set
  Qdrant URL: https://1af9eb0e-770d-400a-8855-b7607c63...
  Sample docs present
  Day 1 helpers imported: parse_*, chunk_recursive, scrub_pii_regex, METADATA_FIELDS


---

## Cell 2 — The 9-field metadata schema

Every chunk gets 9 pieces of metadata attached. The schema is programme-standard
— once you commit to it, downstream code (retrieval, filtering, observability)
can rely on it.

Let's build a chunk with all 9 fields on the sample PDF's page 1.

In [8]:
%pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
from datetime import datetime, timezone

# Print the schema — the fields
print("Metadata schema — 9 fields per chunk:")
for i, field in enumerate(METADATA_FIELDS, 1):
    print(f"  {i}. {field}")

# Build one real chunk with all 9 fields populated
pdf_text = parse_pdf(SAMPLE_PDF)
chunks = chunk_recursive(pdf_text, max_size=400)
first_chunk_text = chunks[0]

chunk_with_metadata = {
    "chunk_id":     "company_policies#0",
    "source":       SAMPLE_PDF.name,
    "doc_type":     "pdf",
    "section_path": "Employee Handbook > Leave Policy",   # from headings
    "page":         1,
    "date":         "2025-01-15",                          # from document metadata
    "language":     "en",
    "version":      "v2.3",
    "ingested_at":  datetime.now(timezone.utc).isoformat(),
    "text":         first_chunk_text,
}

print("\n══ A real chunk with all 9 metadata fields ══\n")
for k, v in chunk_with_metadata.items():
    if k == "text":
        v = v[:80] + "..."
    print(f"  {k:15s}  {v!r}")

Metadata schema — 9 fields per chunk:
  1. chunk_id
  2. source
  3. doc_type
  4. section_path
  5. page
  6. date
  7. language
  8. version
  9. ingested_at



══ A real chunk with all 9 metadata fields ══

  chunk_id         'company_policies#0'
  source           'company_policies.pdf'
  doc_type         'pdf'
  section_path     'Employee Handbook > Leave Policy'
  page             1
  date             '2025-01-15'
  language         'en'
  version          'v2.3'
  ingested_at      '2026-11-21T09:29:00.091269+00:00'
  text             'Acme Corp — Employee Handbook\nSection 1: Leave Policy\nEmployees are entitled to ...'


**Why 9 fields?** Each earns its place at retrieval time:

| Field | Enables |
|---|---|
| `chunk_id` | Unique reference; deduplication; citations |
| `source` | "Which document did this come from?" |
| `doc_type` | Route retrieval by format (e.g. skip PDFs for a code query) |
| `section_path` | Explainable citations: "from HR > Leave Policy" |
| `page` | Deep-link into PDFs for user verification |
| `date` | Filter by recency: "only current-year policies" |
| `language` | Multi-lingual corpora need this |
| `version` | Retrieve latest revision; audit trail |
| `ingested_at` | Debug drift: "when did this chunk enter the system?" |

**Today's overhead, tomorrow's payoff.** W9 uses filters heavily; without
metadata now, W9's precision lift isn't possible.

---

## Cell 3 — Qdrant filter DSL

Metadata is only useful if you can filter on it at query time. Let's build a
small Qdrant collection with 6 chunks (mixed doc types + dates), then run
**filtered queries**.

In [10]:
%pip install qdrant-client

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Qdrant imports successful")

Qdrant imports successful


In [12]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, Range,
)

openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

COLLECTION = "wk08_day2_metadata_demo"

# Recreate
try:
    qdrant.delete_collection(COLLECTION)
except Exception:
    pass
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

# Six chunks with varied metadata.
# NOTE: for date-range filtering, Qdrant's Range works on NUMERIC fields.
# We store both the ISO date string (for display) and a numeric year (for filtering).
# This is a common real-world pattern: keep both a human-readable form and
# a filter-friendly form of the same value.
sample_chunks = [
    {"id": 0, "text": "Annual leave is 20 days per year.",         "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 1, "text": "Sick leave is unlimited with doctor's note.", "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 2, "text": "Remote work requires VP approval.",           "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 3, "text": "Legacy leave policy: 15 days per year.",      "source": "old_policies.pdf", "doc_type": "pdf", "date": "2022-03-01", "year": 2022},
    {"id": 4, "text": "Product tour: analytics dashboards.",         "source": "landing.html",   "doc_type": "html", "date": "2025-06-20", "year": 2025},
    {"id": 5, "text": "Onboarding: meet your team in week 1.",       "source": "onboarding.docx", "doc_type": "docx", "date": "2024-11-10", "year": 2024},
]

# Embed and upsert
texts = [c["text"] for c in sample_chunks]
vectors = [item.embedding for item in openai_client.embeddings.create(
    model="text-embedding-3-small", input=texts).data]

points = [
    PointStruct(id=c["id"], vector=v, payload={k: c[k] for k in ["text", "source", "doc_type", "date", "year"]})
    for c, v in zip(sample_chunks, vectors)
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Upserted {len(points)} chunks with metadata.\n")

Upserted 6 chunks with metadata.



In [13]:
from qdrant_client.models import PayloadSchemaType

# Create an index for filtering by document type
qdrant.create_payload_index(
    collection_name=COLLECTION,
    field_name="doc_type",
    field_schema=PayloadSchemaType.KEYWORD,
)

# Create an index for filtering by year
qdrant.create_payload_index(
    collection_name=COLLECTION,
    field_name="year",
    field_schema=PayloadSchemaType.INTEGER,
)

print("Payload indexes created successfully.")

Payload indexes created successfully.


In [14]:
def show_hits(results, label):
    print(f"── {label} ──")
    if not results:
        print("  (no matches)\n")
        return
    for h in results:
        p = h.payload
        print(f"  {h.score:.3f}  {p['source']:20s} ({p['doc_type']}, {p['date']})  {p['text'][:50]}")
    print()

# Query with NO filter — pure semantic
query = "How much leave do I get per year?"
q_vec = openai_client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

results = qdrant.query_points(collection_name=COLLECTION, query=q_vec, limit=3).points
show_hits(results, f"No filter — Q: {query!r}")

# Now filter: only PDFs
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="doc_type", match=MatchValue(value="pdf"))]),
).points
show_hits(results, "Filter: doc_type = 'pdf'")

# Now filter: only docs from 2024 or later (uses the numeric 'year' field)
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="year", range=Range(gte=2024))]),
).points
show_hits(results, "Filter: year >= 2024 (exclude legacy 2022 doc)")

# Cleanup
qdrant.delete_collection(COLLECTION)
print("Cleaned up demo collection.")

── No filter — Q: 'How much leave do I get per year?' ──
  0.616  policies.pdf         (pdf, 2025-01-15)  Annual leave is 20 days per year.
  0.544  old_policies.pdf     (pdf, 2022-03-01)  Legacy leave policy: 15 days per year.
  0.414  policies.pdf         (pdf, 2025-01-15)  Sick leave is unlimited with doctor's note.

── Filter: doc_type = 'pdf' ──
  0.616  policies.pdf         (pdf, 2025-01-15)  Annual leave is 20 days per year.
  0.544  old_policies.pdf     (pdf, 2022-03-01)  Legacy leave policy: 15 days per year.
  0.414  policies.pdf         (pdf, 2025-01-15)  Sick leave is unlimited with doctor's note.

── Filter: year >= 2024 (exclude legacy 2022 doc) ──
  0.616  policies.pdf         (pdf, 2025-01-15)  Annual leave is 20 days per year.
  0.414  policies.pdf         (pdf, 2025-01-15)  Sick leave is unlimited with doctor's note.
  0.182  policies.pdf         (pdf, 2025-01-15)  Remote work requires VP approval.

Cleaned up demo collection.


**Discussion:**
- Without a filter, the legacy 2022 policy came back — because it matches
  the query semantically. Users would be told the wrong policy.
- The date filter (`date >= 2024-01-01`) excludes legacy. This is a common
  W9 pattern.
- Qdrant's filter DSL supports `must`, `must_not`, `should` (like Elasticsearch),
  and conditions on any payload field. See Qdrant docs for the full grammar.

**Why this matters:** metadata + filters is how you go from 'retrieval sometimes
returns bad results' to 'retrieval respects business rules.'

---

## Cell 4 — Why PII in vector stores is a problem

**Vector stores are search indexes.** Anything you put in them is discoverable
by anyone with query access. If your ingested chunks contain PII, that PII is
one query away from being surfaced.

**Let's watch this happen.** We'll embed a chunk with PII and retrieve it.

In [18]:
# A chunk with real-looking PII (all fake, but the model doesn't know that)
leaky_chunk = (
    "For questions about payroll, contact Michael Torres in Finance. "
    "His email is michael.torres@acme.com and his employee ID is EMP-04521."
)

# Embed it
vec = openai_client.embeddings.create(
    model="text-embedding-3-small", input=[leaky_chunk]).data[0].embedding

# Build a tiny collection with just this one chunk
LEAK_COLL = "wk08_day2_leak_demo"
try: qdrant.delete_collection(LEAK_COLL)
except Exception: pass

qdrant.create_collection(collection_name=LEAK_COLL,
                        vectors_config=VectorParams(size=1536, distance=Distance.COSINE))
qdrant.upsert(collection_name=LEAK_COLL,
             points=[PointStruct(id=0, vector=vec, payload={"text": leaky_chunk})])

# Query with something innocent-looking
query = "who handles payroll questions?"
q_vec = openai_client.embeddings.create(
    model="text-embedding-3-small", input=[query]).data[0].embedding

hits = qdrant.query_points(collection_name=LEAK_COLL, query=q_vec, limit=1).points

print(f"Q: {query!r}\n")
print("Retrieved chunk:")
print(f"  {hits[0].payload['text']}")
print()
print("The PII (name, email, employee ID) came back in the search result.")
print("Anyone with query access to this collection can retrieve it.")
print("That includes future features you haven't built yet.")

# Cleanup
qdrant.delete_collection(LEAK_COLL)

Q: 'who handles payroll questions?'

Retrieved chunk:
  For questions about payroll, contact Michael Torres in Finance. His email is michael.torres@acme.com and his employee ID is EMP-04521.

The PII (name, email, employee ID) came back in the search result.
Anyone with query access to this collection can retrieve it.
That includes future features you haven't built yet.


True

**The rule:** redact BEFORE you ingest, not after.

Once PII is in the vector store, you can't recall it. You'd have to identify
the offending chunks, delete them, and re-ingest cleaned versions. That's
expensive and error-prone.

**Scrub at ingestion time.** Every time.

---

## Cell 5 — PII detection #1: Regex baseline

**Simplest approach:** regular expressions for common patterns — emails, phone
numbers, employee IDs. Fast, no dependencies, but catches only what you write
patterns for.

The `wk08_pipeline.py` helper has 3 patterns built in.

In [19]:
# The patterns we have
print("Regex patterns:")
for label, pattern in PII_PATTERNS.items():
    print(f"  {label:8s}  {pattern.pattern}")

print()

# Test on a paragraph with mixed PII
test_text = (
    "Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or "
    "+1-555-0142. Her employee ID is EMP-01847. You can also reach Michael "
    "Torres at michael.torres@acme.com."
)

scrubbed, flags = scrub_pii_regex(test_text)

print(f"Original text:\n  {test_text}\n")
print(f"Scrubbed text:\n  {scrubbed}\n")
print(f"PII items detected ({len(flags)}):")
for f in flags:
    print(f"  ✓ {f}")

print("\nMissed: names ('Sarah Chen', 'Michael Torres'). Regex can't do NER.")
print("That's what Presidio brings — next cell.")

Regex patterns:
  EMAIL     \b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b
  PHONE     \+?\d{0,2}[-.\s]?\(?\d{3}\)?[-.\s]?\d{3,4}(?:[-.\s]?\d{4})?\b
  EMP_ID    \bEMP-\d{4,6}\b

Original text:
  Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or +1-555-0142. Her employee ID is EMP-01847. You can also reach Michael Torres at michael.torres@acme.com.

Scrubbed text:
  Contact Sarah Chen (HR Business Partner) at [EMAIL] or [PHONE]. Her employee ID is [EMP_ID]. You can also reach Michael Torres at [EMAIL].

PII items detected (4):
  ✓ EMAIL: sarah.chen@acme.com
  ✓ EMAIL: michael.torres@acme.com
  ✓ PHONE: +1-555-0142
  ✓ EMP_ID: EMP-01847

Missed: names ('Sarah Chen', 'Michael Torres'). Regex can't do NER.
That's what Presidio brings — next cell.


---

## Cell 6 — PII detection #2: Presidio NER

**Presidio** uses named entity recognition (NER, backed by spaCy) to find
entities that regex can't — especially **person names, locations, and
organisations**.

It also handles emails, phones, and IDs — with confidence scores per detection.

In [20]:
# Presidio was imported and asserted in Cell 1
analyzer = AnalyzerEngine()

# Analyze the same text as Cell 5
results = analyzer.analyze(text=test_text, language="en")

print(f"Text:\n  {test_text}\n")
print(f"Presidio detected {len(results)} entities:\n")
print(f"  {'ENTITY_TYPE':<20s} {'SCORE':<7s} {'TEXT':<30s}")
print(f"  {'-'*20} {'-'*7} {'-'*30}")

for r in sorted(results, key=lambda x: x.start):
    entity_text = test_text[r.start:r.end]
    print(f"  {r.entity_type:<20s} {r.score:<7.2f} {entity_text:<30s}")

print()
print("Notice PERSON entities — regex could NEVER find these.")
print("This is why Presidio (or a similar NER tool) is non-negotiable for high-recall PII.")

Text:
  Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or +1-555-0142. Her employee ID is EMP-01847. You can also reach Michael Torres at michael.torres@acme.com.

Presidio detected 8 entities:

  ENTITY_TYPE          SCORE   TEXT                          
  -------------------- ------- ------------------------------
  PERSON               0.85    Sarah Chen                    
  EMAIL_ADDRESS        1.00    sarah.chen@acme.com           
  URL                  0.50    sarah.ch                      
  URL                  0.50    acme.com                      
  PERSON               0.85    Michael Torres                
  EMAIL_ADDRESS        1.00    michael.torres@acme.com       
  URL                  0.50    michael.to                    
  URL                  0.50    acme.com                      

Notice PERSON entities — regex could NEVER find these.
This is why Presidio (or a similar NER tool) is non-negotiable for high-recall PII.


**Discussion:**
- Did Presidio catch 'Sarah Chen' and 'Michael Torres'? (It should.)
- What about the emails and phone numbers — did they show up too? (Yes, Presidio
  has its own regexes internally, plus NER on top.)
- The `score` column matters — 0.85+ is high confidence, 0.5-0.85 medium, below
  0.5 is worth double-checking manually.

**Presidio isn't perfect.** It misses:
- Non-Latin names (e.g. Chinese, Arabic scripts)
- Novel ID formats (custom internal codes)
- Addresses in unusual layouts

Which is why the three-legs pattern (Cell 8) matters.

---

## Cell 7 — Presidio anonymization

**Detection is half the job.** The other half is doing something with the
detections — usually replacing them with placeholders.

Presidio's `AnonymizerEngine` takes the analyzer output and applies
replacements.

In [21]:
anonymizer = AnonymizerEngine()

# Analyze then anonymize
results = analyzer.analyze(text=test_text, language="en")
anonymized = anonymizer.anonymize(text=test_text, analyzer_results=results)

print("Before:")
print(f"  {test_text}\n")
print("After anonymization:")
print(f"  {anonymized.text}\n")
print(f"Replacements made: {len(anonymized.items)}")

Before:
  Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or +1-555-0142. Her employee ID is EMP-01847. You can also reach Michael Torres at michael.torres@acme.com.

After anonymization:
  Contact <PERSON> (HR Business Partner) at <EMAIL_ADDRESS> or +1-555-0142. Her employee ID is EMP-01847. You can also reach <PERSON> at <EMAIL_ADDRESS>.

Replacements made: 4


**Default behaviour:** replace with `<ENTITY_TYPE>` tokens like `<PERSON>`
or `<EMAIL_ADDRESS>`. You can customise operators — hash, encrypt, redact-
with-length-preserved, etc. Presidio docs cover the details.

**For RAG:** the default replacement is fine. The chunk 'contact <PERSON> at
<EMAIL_ADDRESS>' is still retrievable for 'who do I contact?' queries, but
the actual identity isn't leaked.

---

## Cell 8 — The three legs: regex + Presidio + manual audit

**No single tool catches everything.** Production PII pipelines have three legs:

1. **Regex** — fast; catches obvious patterns (emails, phones, IDs); low recall on novel formats
2. **Presidio (or similar NER)** — catches names, orgs, locations; higher recall; slower; misses non-Latin scripts
3. **Manual audit** — human eyes on a sample of ingested chunks; catches what both missed

Let's build a hybrid `scrub_pii` that runs both regex and Presidio, then
returns everything flagged.

In [22]:
def scrub_pii_hybrid(text: str) -> tuple[str, list[dict]]:
    """Regex + Presidio combined. Returns (scrubbed_text, flags).
    
    flags is a list of dicts: {source, entity_type, matched_text, score}
    """
    flags = []
    
    # Leg 1: regex scrub (from Day 1 helper)
    scrubbed, regex_flags = scrub_pii_regex(text)
    for f in regex_flags:
        entity_type, matched = f.split(": ", 1)
        flags.append({"source": "regex", "entity_type": entity_type,
                     "matched_text": matched, "score": 1.0})
    
    # Leg 2: Presidio on the ALREADY-regex-scrubbed text
    # (avoids double-flagging emails/phones that regex already redacted)
    results = analyzer.analyze(text=scrubbed, language="en")
    if results:
        anonymized = anonymizer.anonymize(text=scrubbed, analyzer_results=results)
        scrubbed = anonymized.text
        for r in results:
            flags.append({"source": "presidio", "entity_type": r.entity_type,
                         "matched_text": text[r.start:r.end], "score": r.score})
    
    return scrubbed, flags

# Run on the test text
scrubbed, flags = scrub_pii_hybrid(test_text)

print(f"Original:\n  {test_text}\n")
print(f"Scrubbed:\n  {scrubbed}\n")
print(f"Total flags: {len(flags)}\n")

print(f"  {'SOURCE':<10s} {'ENTITY':<20s} {'SCORE':<7s} {'MATCHED':<25s}")
print(f"  {'-'*10} {'-'*20} {'-'*7} {'-'*25}")
for f in flags:
    print(f"  {f['source']:<10s} {f['entity_type']:<20s} {f['score']:<7.2f} {f['matched_text']:<25s}")

print("\n**Leg 3 (audit):** in production, human eyes review 10-20 random chunks")
print("per ingestion run. Look for what neither tool caught. Log gaps in an audit doc.")

Original:
  Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or +1-555-0142. Her employee ID is EMP-01847. You can also reach Michael Torres at michael.torres@acme.com.

Scrubbed:
  Contact <PERSON> (HR Business Partner) at [EMAIL] or [PHONE]. Her employee ID is [EMP_ID]. You can also reach <PERSON> at [EMAIL].

Total flags: 6

  SOURCE     ENTITY               SCORE   MATCHED                  
  ---------- -------------------- ------- -------------------------
  regex      EMAIL                1.00    sarah.chen@acme.com      
  regex      EMAIL                1.00    michael.torres@acme.com  
  regex      PHONE                1.00    +1-555-0142              
  regex      EMP_ID               1.00    EMP-01847                
  presidio   PERSON               0.85    Sarah Chen               
  presidio   PERSON               0.85    u can also rea           

**Leg 3 (audit):** in production, human eyes review 10-20 random chunks
per ingestion run. Look for what neith

**Real-world reality check:**
- Regex + Presidio is a strong pairing — usually 90-95% recall on English text
- The remaining 5-10% needs human review for high-stakes data
- **Don't skip audit.** It's tedious but catches novel PII patterns your tools
  don't know about (custom ID formats, unusual addresses, initials-only names)

---

## Cell 9 — Data residency: name your stack

**Every RAG system spans jurisdictions.** Your embeddings live somewhere, your
LLM call goes somewhere, your vector store is hosted somewhere. Each hop crosses
borders. GDPR, DPDP, and similar regulations care about this.

**W8's residency requirement:** you can articulate your stack in one sentence.

Not a deep-dive — that's W28. Just name it.

In [23]:
def describe_stack() -> None:
    """Print a one-sentence description of where your data lives."""
    qdrant_url = os.environ.get("QDRANT_URL", "unknown")
    
    # Detect Qdrant Cloud region from URL if possible
    if "aws" in qdrant_url.lower():
        qdrant_region = "AWS (region in URL)"
    elif "gcp" in qdrant_url.lower():
        qdrant_region = "Google Cloud (region in URL)"
    elif "azure" in qdrant_url.lower():
        qdrant_region = "Azure (region in URL)"
    elif "localhost" in qdrant_url or "127.0.0.1" in qdrant_url:
        qdrant_region = "local machine"
    else:
        qdrant_region = "Qdrant Cloud (check dashboard for region)"
    
    print("═══ YOUR DATA RESIDENCY STACK ═══\n")
    print(f"  Embeddings computed by: OpenAI API (US)")
    print(f"  Vectors stored at:      {qdrant_region}")
    print(f"                          URL: {qdrant_url[:50]}...")
    print(f"  LLM inference at:       OpenAI API (US)")
    print(f"  App layer:              wherever you run this notebook")
    print()
    print("One-sentence articulation:")
    print("  \"My embeddings are computed and my LLM calls are served by OpenAI (US);")
    print("   my vectors sit in Qdrant Cloud; my source documents live locally.\"\n")
    print("Jurisdictions crossed: US (OpenAI), + Qdrant region, + wherever your app runs.")
    print("If your users' data is EU-residents' data → GDPR applies. Log it in your ADR.")

describe_stack()

═══ YOUR DATA RESIDENCY STACK ═══

  Embeddings computed by: OpenAI API (US)
  Vectors stored at:      AWS (region in URL)
                          URL: https://1af9eb0e-770d-400a-8855-b7607c63a696.eu-ce...
  LLM inference at:       OpenAI API (US)
  App layer:              wherever you run this notebook

One-sentence articulation:
  "My embeddings are computed and my LLM calls are served by OpenAI (US);
   my vectors sit in Qdrant Cloud; my source documents live locally."

Jurisdictions crossed: US (OpenAI), + Qdrant region, + wherever your app runs.
If your users' data is EU-residents' data → GDPR applies. Log it in your ADR.


**That's the W8 residency bar.** Not a full compliance analysis — just being
able to answer 'where does my data go?' for your capstone stack.

W28 deep-dives residency, retention, right-to-be-forgotten, and audit logging.
For now, put your stack sentence in your ADR (Track B step 4).

---

## Cell 10 — Wire it together: mini-pipeline on ONE document

**The synthesis moment.** Take one sample document, run it end-to-end through:
1. Parse (Day 1)
2. Structure-aware chunk (Day 1)
3. Scrub PII (Cell 8 — hybrid regex + Presidio)
4. Attach metadata (Cell 2 — 9 fields)
5. Print the final list of ingestion-ready chunks

We do NOT push these to Qdrant here — that's Track B on the capstone corpus.
Today's goal is to see the shape of an ingested chunk.

In [24]:
from datetime import datetime, timezone

def ingest_one_pdf(path: Path) -> list[dict]:
    """End-to-end ingestion of a single PDF: parse → chunk → scrub → enrich.
    
    Returns a list of dicts ready for Qdrant upsert.
    """
    # Step 1: parse
    text = parse_pdf(path)
    
    # Step 2: chunk (recursive is the sane default from Day 1)
    chunk_texts = chunk_recursive(text, max_size=400)
    
    # Step 3: scrub PII from each chunk
    ingested_at = datetime.now(timezone.utc).isoformat()
    doc_stem = path.stem  # e.g. 'company_policies'
    
    ingested = []
    for idx, raw_text in enumerate(chunk_texts):
        scrubbed, flags = scrub_pii_hybrid(raw_text)
        
        # Step 4: enrich with metadata
        chunk = {
            "chunk_id":         f"{doc_stem}#{idx}",
            "source":           path.name,
            "doc_type":         path.suffix.lstrip(".").lower(),
            "section_path":     doc_stem,  # placeholder; structure-aware would fill this properly
            "page":             None,       # PDF-parse could extract this per-chunk if needed
            "date":             "2025-01-15",  # in reality, extract from PDF metadata
            "language":         "en",
            "version":          "v1",
            "ingested_at":      ingested_at,
            "text":             scrubbed,
            "pii_flags_count":  len(flags),
        }
        ingested.append(chunk)
    
    return ingested

# Run on the sample PDF
chunks = ingest_one_pdf(SAMPLE_PDF)

print(f"Ingested {SAMPLE_PDF.name} → {len(chunks)} chunks\n")
print(f"Total PII items scrubbed: {sum(c['pii_flags_count'] for c in chunks)}\n")
print("══ Sample of ingested chunks ══\n")

for c in chunks[:3]:  # first 3 for brevity
    print(f"[{c['chunk_id']}]")
    print(f"  source: {c['source']}, doc_type: {c['doc_type']}, pii_flags: {c['pii_flags_count']}")
    print(f"  text:   {c['text'][:120]}...")
    print()

# Show a chunk that has PII scrubbed (should be the HR contact one)
pii_chunks = [c for c in chunks if c['pii_flags_count'] > 0]
if pii_chunks:
    print(f"══ Example of a scrubbed chunk (had {pii_chunks[0]['pii_flags_count']} PII items) ══")
    print(f"  {pii_chunks[0]['text']}")
    print("\n  Notice: emails, phones, names all replaced with placeholders.")
    print("  The chunk is still semantically useful for retrieval, but the PII is gone.")

Ingested company_policies.pdf → 7 chunks

Total PII items scrubbed: 29

══ Sample of ingested chunks ══

[company_policies#0]
  source: company_policies.pdf, doc_type: pdf, pii_flags: 4
  text:   Acme Corp — Employee Handbook
Section 1: Leave Policy
Employees are entitled to <DATE_TIME> of paid <DATE_TIME> leave pe...

[company_policies#1]
  source: company_policies.pdf, doc_type: pdf, pii_flags: 7
  text:   Sick leave is separate and unlimited, subject to a doctor's note for absences longer than three
consecutive <DATE_TIME>....

[company_policies#2]
  source: company_policies.pdf, doc_type: pdf, pii_flags: 1
  text:   Section 3: Remote Work Policy
All full-time employees are eligible for hybrid work arrangements. The standard expectatio...

══ Example of a scrubbed chunk (had 4 PII items) ══
  Acme Corp — Employee Handbook
Section 1: Leave Policy
Employees are entitled to <DATE_TIME> of paid <DATE_TIME> leave per calendar year. Leave must be requested at
least two weeks in advance th

**That's what a chunk looks like at ingestion time.** All 9 metadata fields
populated, PII scrubbed, text ready for embedding + upsert to Qdrant.

**Not shown here (Track B does):**
- Wrapping `ingest_one_pdf` into `ingest_corpus(dir)` that dispatches by extension
- Embedding the scrubbed text
- Upserting to Qdrant with the full metadata payload
- Running the golden set against the new collection

**Track B is where this shape becomes real capstone code.**

---

## Cell 11 — Wrap: what you built across two days

**Day 1 — Parsing + chunking:**
1. PyMuPDF (fast), pdfplumber (tables), BeautifulSoup (strip nav/footer), python-docx (styles)
2. The scanned-PDF trap — empty text, silent failure
3. Four chunking strategies compared; structure-aware wins when documents have headings

**Day 2 — Metadata + PII + wiring:**
4. 9-field metadata schema — each field earns retrieval-time payoff
5. Qdrant filter DSL — filter by doc_type, date, source, etc.
6. PII in vector stores is discoverable — scrub at ingestion, not after
7. Regex + Presidio + audit — three legs for high-recall PII
8. Data residency — you can articulate your stack in one sentence
9. Mini-pipeline: parse → chunk → scrub → enrich → ingestion-ready chunk

**Track B (take-home): apply this to your capstone corpus.**

Open `AI-RAG_W8_Application_Growth_Guide.md`. You will:
- Add `src/ingest/pipeline.py` (~120 lines) — the full corpus ingester
- Create Qdrant collection `capstone_chunks_v2` with metadata payload
- Re-ingest your capstone corpus
- Verify PII scrubbing on 10 random chunks (audit doc)
- Re-run golden set, compare wk8 vs wk7 snapshot
- Update ADR with ingestion decisions

About 3 hours self-paced.

---

**Heads-up for W9:** the difficulty heatmap flags W9 as the **first DANGER
ZONE** — hybrid search + reranking + query rewriting all at once. Plan extra
lab time. But your W8 metadata will make W9's filtered retrieval powerful.

# Week8 Practice Assignment

In [25]:
import os
print(os.getcwd())

/voc/work/Demo Files/demos


In [26]:
from pathlib import Path

stress_file = Path("data/stress_test/pii_adversarial.md")

print("File exists:", stress_file.exists())

if stress_file.exists():
    print(stress_file.read_text(encoding="utf-8")[:500])

File exists: True
```markdown
# Internal Memo — Project Khanjar

## Project team

Please direct queries to **Dr. Yuki Tanaka** (Tokyo office),
**Fatima Al-Rashid** (Dubai office), or **Rajesh Patel** (Mumbai
office). Aisha Mohammed will cover the European leg.

For US contacts: Carlos Mendoza is the regional lead — you can
reach him via the Atlanta office.

## Reaching out

You can email the team lead at alice [at] example [dot] com
(spaces matter — the form scrapes a lot of bots otherwise).

Alternative inbox is


# Step 1a: Read our adversarial test document
What this code does
Step	What happens
1	Reads your pii_adversarial.md file
2	Splits the document into chunks of approximately 800 characters
3	Sends each chunk through your existing hybrid scrubber
4	Displays detected PII and the scrubbed text
5	Counts the total detection flags

One important distinction: Detection flags are not necessarily the same as unique PII items. We will inspect the output before calculating how many items were caught or missed.

In [27]:
from pathlib import Path

# Step 1: Read our adversarial test document
stress_file = Path("data/stress_test/pii_adversarial.md")
raw_text = stress_file.read_text(encoding="utf-8")

print("Original document length:", len(raw_text))


# Step 2: Split the document into chunks
stress_chunks = chunk_recursive(raw_text, max_size=800)

print("Number of chunks:", len(stress_chunks))


# Step 3: Apply the existing hybrid scrubber to each chunk
stress_results = []
total_flags = 0

for i, chunk in enumerate(stress_chunks):

    scrubbed_text, flags = scrub_pii_hybrid(chunk)

    stress_results.append({
        "chunk_number": i + 1,
        "original_text": chunk,
        "scrubbed_text": scrubbed_text,
        "flags": flags
    })

    total_flags += len(flags)

    print(f"\n{'='*60}")
    print(f"CHUNK {i + 1}")
    print(f"{'='*60}")

    print("\nDetected PII:")
    for flag in flags:
        print(
            flag["source"],
            "|",
            flag["entity_type"],
            "|",
            flag["matched_text"],
            "| Score:",
            flag["score"]
        )

    print("\nScrubbed text:")
    print(scrubbed_text)


print("\n" + "="*60)
print("STRESS TEST SUMMARY")
print("="*60)
print("Total chunks:", len(stress_chunks))
print("Total detection flags:", total_flags)

Original document length: 1741
Number of chunks: 21

CHUNK 1

Detected PII:

Scrubbed text:
```markdown
# Internal Memo — Project Khanjar

CHUNK 2

Detected PII:

Scrubbed text:
## Project team

CHUNK 3

Detected PII:
presidio | PERSON | Yuki Tanaka | Score: 0.85
presidio | LOCATION | Tokyo | Score: 0.85
presidio | PERSON | Fatima Al-Rashid | Score: 0.85
presidio | LOCATION | Dubai | Score: 0.85
presidio | PERSON | Rajesh Patel | Score: 0.85
presidio | LOCATION | Mumbai | Score: 0.85
presidio | PERSON | Aisha Mohammed | Score: 0.85
presidio | NRP | European | Score: 0.85

Scrubbed text:
Please direct queries to **Dr. <PERSON>** (<LOCATION> office),
**<PERSON>** (<LOCATION> office), or **<PERSON>** (<LOCATION>
office). <PERSON> will cover the <NRP> leg.

CHUNK 4

Detected PII:
presidio | LOCATION | US | Score: 0.85
presidio | PERSON | Carlos Mendoza | Score: 0.85
presidio | LOCATION | Atlanta | Score: 0.85

Scrubbed text:
For <LOCATION> contacts: <PERSON> is the regional lead — you can


Your stress test has executed successfully. We have our first result:
21
Chunks created

42
Detection flags



The output is truncated, so we cannot yet determine which individual PII items were caught or missed.

One important observation: the first two chunks contain only the Markdown code fence and a heading. They have no detected PII, which is expected. Our next task is to examine the actual detections.

# Step 1b — Display all detections in a compact table

This groups repeated detections of the same item, so we can distinguish unique items from repeated flags.

In [28]:
from collections import defaultdict

# Group detections by their type and matched text
detection_summary = defaultdict(list)

for result in stress_results:
    for flag in result["flags"]:
        key = (
            flag["source"],
            flag["entity_type"],
            flag["matched_text"]
        )
        detection_summary[key].append(result["chunk_number"])

print("TOTAL UNIQUE DETECTIONS:", len(detection_summary))
print("\nDETECTION DETAILS\n")

for (source, entity_type, matched_text), chunks in detection_summary.items():
    print(
        f"Source: {source:10} | "
        f"Type: {entity_type:20} | "
        f"Matched: {matched_text:35} | "
        f"Chunk(s): {chunks}"
    )

TOTAL UNIQUE DETECTIONS: 41

DETECTION DETAILS

Source: presidio   | Type: PERSON               | Matched: Yuki Tanaka                         | Chunk(s): [3]
Source: presidio   | Type: LOCATION             | Matched: Tokyo                               | Chunk(s): [3, 14]
Source: presidio   | Type: PERSON               | Matched: Fatima Al-Rashid                    | Chunk(s): [3]
Source: presidio   | Type: LOCATION             | Matched: Dubai                               | Chunk(s): [3]
Source: presidio   | Type: PERSON               | Matched: Rajesh Patel                        | Chunk(s): [3]
Source: presidio   | Type: LOCATION             | Matched: Mumbai                              | Chunk(s): [3]
Source: presidio   | Type: PERSON               | Matched: Aisha Mohammed                      | Chunk(s): [3]
Source: presidio   | Type: NRP                  | Matched: European                            | Chunk(s): [3]
Source: presidio   | Type: LOCATION             | Matched: U

# Step 2 — Display the scrubbed text only for chunks containing detections

This lets us see whether the PII was actually replaced or merely detected.
What we will analyse next
We'll compare the results against all six categories in your assignment:
1. Names
2. Obfuscated email addresses
3. Phone numbers
4. Employee identifiers
5. Physical addresses
6. Financial information

In [29]:
for result in stress_results:
    if result["flags"]:
        print(f"\n{'='*60}")
        print("CHUNK:", result["chunk_number"])
        print(f"{'='*60}")
        print(result["scrubbed_text"])


CHUNK: 3
Please direct queries to **Dr. <PERSON>** (<LOCATION> office),
**<PERSON>** (<LOCATION> office), or **<PERSON>** (<LOCATION>
office). <PERSON> will cover the <NRP> leg.

CHUNK: 4
For <LOCATION> contacts: <PERSON> is the regional lead — you can
reach him via the <LOCATION> office.

CHUNK: 6
You can email the team lead at <PERSON> [at] example [dot] com
(spaces matter — the form scrapes a lot of bots otherwise).

CHUNK: 7
Alternative inbox is <URL>ith ＠ example ．com (the at-sign and
dot are full-width <NRP> characters, copy carefully).

CHUNK: 10
- Yuki's office: +81 (0) 3 [PHONE]
- <PERSON>'s mobile: zero zero nine seven one five zero two three four five six seven
- Rajesh's desk: 91-22-[PHONE] ext. 4501
- Aisha's WhatsApp: +44 [PHONE] 123
- <PERSON>'s direct: <PHONE_NUMBER>

CHUNK: 12
<PERSON> — [EMP_ID]
- <PERSON> — Staff #4082
- <PERSON> — RJP/2019/0451
- Aisha Mohammed — A.M.-2021-EUR-007
- <PERSON> — Badge: <US_DRIVER_LICENSE>

CHUNK: 14
- <LOCATION> office: 1-2-3 Shibuya

Your results reveal something important: the hybrid scrubber is detecting PII, but it is also making mistakes. In fact, this is exactly what the Week 8 assignment wants us to investigate.

Let's interpret the results before preparing your findings document.
1. What your results tell us
42
Total detection flags

41
Unique detections


The difference is one repeated detection: Tokyo was detected in two chunks.

A. Names
Name	Observed result
Yuki Tanaka	Caught
Fatima Al-Rashid	Caught
Rajesh Patel	Caught
Aisha Mohammed	Caught
Carlos Mendoza	Caught
Alice	False positive
Bob Smith	Not detected as a person in the visible results

Presidio successfully identified the five full names. However, it incorrectly classified alice in the obfuscated email as a person.
B. Obfuscated email addresses
Email format	Observed result
alice [at] example [dot] com	Not recognised as an email; alice was incorrectly classified as PERSON
bob.smith ＠ example ．com	Part of the address was incorrectly classified as URL
y.tanaka(at)hospital(dot)org	No matching email detection visible

This demonstrates that ordinary email detection does not automatically handle deliberately obfuscated formats.
C. Phone numbers
Your regex caught some parts of the phone numbers:
- 1234 5678
- 2570-1234
- 7700 900
Presidio also detected part of Carlos's spaced phone number.

However, some detections are incomplete. For example, 7700 900 is only part of Aisha's phone number.
D. Employee identifiers
Identifier	Observed result
EMP-04827	Caught by regex
Staff #4082	Not detected in visible results
RJP/2019/0451	Not detected in visible results
A.M.-2021-EUR-007	Not detected in visible results
Badge: CM18234	Partially detected by Presidio as CM1823

Your existing regex has a specific EMP-ID pattern. It does not cover the other organisation-specific formats.
E. Addresses
Presidio detected several location-related items, including Tokyo, Dubai and Mumbai.

However, it also detected incomplete fragments such as , Jap, - Dub, and : 221B Baker Stre.

This suggests that some address information is being detected incorrectly or only partially.
F. Financial information
The visible output shows:
- Part of the IBAN was incorrectly classified as a phone number.
- The project code was not detected in the visible results, which is desirable because it is a distractor rather than PII.
- I cannot yet confirm whether the masked card number was caught.
2. One issue we should investigate before writing the findings
Your document was split into 21 chunks. Some detections appear to contain incomplete text, such as:
- - Fati
- 23 - C
- Fatima Al-Rashi
- Rajesh Pate
This matters because a chunk boundary can split a name, phone number or address. A scrubber may then fail to recognise the complete item.

We should check whether the original PII strings remain in the scrubbed output.

# 3. Run this verification cell

How to interpret this test:
- STILL PRESENT means the complete original string remains in the scrubbed output.
- CHANGED / NOT PRESENT means the exact original string is no longer present. It does not automatically prove complete redaction; we must inspect partial leftovers.

In [30]:
# Combine all scrubbed chunks
all_scrubbed_text = "\n".join(
    result["scrubbed_text"]
    for result in stress_results
)

# Items we want to verify
pii_items = {
    "Names": [
        "Yuki Tanaka",
        "Fatima Al-Rashid",
        "Rajesh Patel",
        "Aisha Mohammed",
        "Carlos Mendoza"
    ],

    "Emails": [
        "alice [at] example [dot] com",
        "bob.smith ＠ example ．com",
        "y.tanaka(at)hospital(dot)org"
    ],

    "Phones": [
        "+81 (0) 3 1234 5678",
        "zero zero nine seven one five zero two three four five six seven",
        "91-22-2570-1234 ext. 4501",
        "+44 7700 900 123",
        "4 0 4 . 5 5 5 . 0 1 8 2"
    ],

    "Employee IDs": [
        "EMP-04827",
        "Staff #4082",
        "RJP/2019/0451",
        "A.M.-2021-EUR-007",
        "Badge: CM18234"
    ],

    "Addresses": [
        "1-2-3 Shibuya-ku, Tokyo 150-0002, Japan",
        "Office 412, Sheikh Zayed Road, Dubai, UAE",
        "Flat 4B, 23rd Cross Road, Bandra West, Mumbai 400050",
        "221B Baker Street, London NW1 6XE"
    ],

    "Financial": [
        "PRJ-2024-KHANJAR-EU",
        "GB82 WEST 1234 5698 7654 32",
        "****-****-****-4521"
    ]
}

for category, items in pii_items.items():
    print(f"\n{'='*55}")
    print(category)
    print('='*55)

    for item in items:
        status = (
            "STILL PRESENT"
            if item in all_scrubbed_text
            else "CHANGED / NOT PRESENT"
        )

        print(f"{status:25} | {item}")


Names
CHANGED / NOT PRESENT     | Yuki Tanaka
CHANGED / NOT PRESENT     | Fatima Al-Rashid
CHANGED / NOT PRESENT     | Rajesh Patel
STILL PRESENT             | Aisha Mohammed
CHANGED / NOT PRESENT     | Carlos Mendoza

Emails
CHANGED / NOT PRESENT     | alice [at] example [dot] com
CHANGED / NOT PRESENT     | bob.smith ＠ example ．com
STILL PRESENT             | y.tanaka(at)hospital(dot)org

Phones
CHANGED / NOT PRESENT     | +81 (0) 3 1234 5678
STILL PRESENT             | zero zero nine seven one five zero two three four five six seven
CHANGED / NOT PRESENT     | 91-22-2570-1234 ext. 4501
CHANGED / NOT PRESENT     | +44 7700 900 123
CHANGED / NOT PRESENT     | 4 0 4 . 5 5 5 . 0 1 8 2

Employee IDs
CHANGED / NOT PRESENT     | EMP-04827
STILL PRESENT             | Staff #4082
STILL PRESENT             | RJP/2019/0451
STILL PRESENT             | A.M.-2021-EUR-007
CHANGED / NOT PRESENT     | Badge: CM18234

Addresses
CHANGED / NOT PRESENT     | 1-2-3 Shibuya-ku, Tokyo 150-0002, Japan
CHAN

This verification gives us a much clearer picture. We can now distinguish between information that was completely missed and information that may have been only partially redacted.

There is also an important lesson here: **a string disappearing from the output does not necessarily mean all its sensitive information was removed.**

## **1. Verification results**

I've classified the results below using the exact-string check you performed.

### **Names**

| Name             | Exact original string remains? |
| ---------------- | ------------------------------ |
| Yuki Tanaka      | No                             |
| Fatima Al-Rashid | No                             |
| Rajesh Patel     | No                             |
| Aisha Mohammed   | **Yes**                        |
| Carlos Mendoza   | No                             |

Four names were altered, but Aisha Mohammed remains completely visible.

### **Emails**

| Email                        | Exact original string remains? |
| ---------------------------- | ------------------------------ |
| alice [at] example [dot] com | No                             |
| bob.smith ＠ example ．com     | No                             |
| y.tanaka(at)hospital(dot)org | **Yes**                        |

The first two were altered, but we need to check whether their complete addresses were actually protected.

### **Phone numbers**

| Phone                            | Exact original string remains? |
| -------------------------------- | ------------------------------ |
| +81 (0) 3 1234 5678              | No                             |
| Spelled-out international number | **Yes**                        |
| 91-22-2570-1234 ext. 4501        | No                             |
| +44 7700 900 123                 | No                             |
| 4 0 4 . 5 5 5 . 0 1 8 2          | No                             |

The spelled-out number was completely missed.

### **Employee identifiers**

| Identifier        | Exact original string remains? |
| ----------------- | ------------------------------ |
| EMP-04827         | No                             |
| Staff #4082       | **Yes**                        |
| RJP/2019/0451     | **Yes**                        |
| A.M.-2021-EUR-007 | **Yes**                        |
| Badge: CM18234    | No                             |

Three identifiers remain completely visible.

### **Addresses**

All four complete original address strings are no longer present. However, we cannot yet conclude that the complete addresses were redacted correctly. Some Presidio detections were only fragments.

### **Financial information**

| Item         | Exact original string remains? |
| ------------ | ------------------------------ |
| Project code | **Yes**                        |
| IBAN         | No                             |
| Masked card  | **Yes**                        |

The project code should remain visible because it is a distractor, not PII. The masked card remains visible and should be recorded as a missed item.

## **2. What we have learned so far**

Your experiment demonstrates four important limitations:

1. **Incomplete recognition:** Some formats, such as spelled-out phone numbers and unusual employee IDs, are not recognised.
2. **Partial detection:** A phone number or address may be detected only in fragments.
3. **False positives:** Presidio classified words such as `alice` and `European` as entities even though they were not necessarily PII.
4. **Different detection and protection outcomes:** Finding a fragment does not guarantee that the entire sensitive value is removed.

## **3. One final audit before preparing your findings document**

I want to verify the partial redactions. This is particularly important for the addresses, IBAN, and the items that were changed but may still expose sensitive fragments.

Add this cell to your existing notebook:

This time, we want to inspect the complete original and scrubbed text together.

Why this matters: Suppose the original IBAN is:

GB82 WEST 1234 5698 7654 32

If the scrubber replaces only 1234 5698 7654, the exact original IBAN disappears, but the remaining fragments still expose sensitive information.

Once you share this output, we'll complete the findings table with three separate outcomes: fully redacted, partially redacted, and missed. Then I'll help you create the required docs/adr/pii_stress_test_findings.md file using your actual experiment results.

In [32]:
# Display original and scrubbed versions of every chunk
# that contains at least one detection.

for result in stress_results:

    if result["flags"]:

        print("\n" + "=" * 70)
        print("CHUNK:", result["chunk_number"])
        print("=" * 70)

        print("\nORIGINAL TEXT:\n")
        print(result["original_text"])

        print("\nSCRUBBED TEXT:\n")
        print(result["scrubbed_text"])


CHUNK: 3

ORIGINAL TEXT:

Please direct queries to **Dr. Yuki Tanaka** (Tokyo office),
**Fatima Al-Rashid** (Dubai office), or **Rajesh Patel** (Mumbai
office). Aisha Mohammed will cover the European leg.

SCRUBBED TEXT:

Please direct queries to **Dr. <PERSON>** (<LOCATION> office),
**<PERSON>** (<LOCATION> office), or **<PERSON>** (<LOCATION>
office). <PERSON> will cover the <NRP> leg.

CHUNK: 4

ORIGINAL TEXT:

For US contacts: Carlos Mendoza is the regional lead — you can
reach him via the Atlanta office.

SCRUBBED TEXT:

For <LOCATION> contacts: <PERSON> is the regional lead — you can
reach him via the <LOCATION> office.

CHUNK: 6

ORIGINAL TEXT:

You can email the team lead at alice [at] example [dot] com
(spaces matter — the form scrapes a lot of bots otherwise).

SCRUBBED TEXT:

You can email the team lead at <PERSON> [at] example [dot] com
(spaces matter — the form scrapes a lot of bots otherwise).

CHUNK: 7

ORIGINAL TEXT:

Alternative inbox is bob.smith ＠ example ．com (the 

The chunk-by-chunk comparison shows that the hybrid scrubber is detecting some PII successfully, but it is also **missing several sensitive details and partially masking others**. This is exactly what the stress test is intended to reveal.

Let's classify your actual results before writing the findings document.

## **1. What the results tell us**

### **A. Names**

Mostly detected

| Original information | Result                                           | Observation                     |
| -------------------- | ------------------------------------------------ | ------------------------------- |
| Dr. Yuki Tanaka      | `Dr. <PERSON>`                                   | Partially masked; title remains |
| Fatima Al-Rashid     | `<PERSON>`                                       | Masked                          |
| Rajesh Patel         | `<PERSON>`                                       | Masked                          |
| Aisha Mohammed       | `<PERSON>` in Chunk 3, but unchanged in Chunk 12 | Inconsistent detection          |
| Carlos Mendoza       | `<PERSON>`                                       | Masked                          |

**Important:** Aisha Mohammed is a clear example of inconsistent results. The scrubber masks her name in Chunk 3 but leaves it untouched in Chunk 12.

### **B. Email addresses**

Not reliably protected

| Original information           | Result                            | Problem                           |
| ------------------------------ | --------------------------------- | --------------------------------- |
| `alice [at] example [dot] com` | `<PERSON> [at] example [dot] com` | Only the name portion was masked  |
| `bob.smith ＠ example ．com`     | `<URL>ith ＠ example ．com`         | Only part of the email was masked |
| `y.tanaka(at)hospital(dot)org` | Still present                     | Completely missed                 |

The scrubber does not understand these alternative email formats. It identifies fragments, but does not remove the complete email address.

### **C. Phone numbers**

Partially detected

| Original information        | Result                    | Problem                                                  |
| --------------------------- | ------------------------- | -------------------------------------------------------- |
| `+81 (0) 3 1234 5678`       | `+81 (0) 3 [PHONE]`       | Country and area code remain                             |
| Spelled-out digits          | Still present             | Completely missed                                        |
| `91-22-2570-1234 ext. 4501` | `91-22-[PHONE] ext. 4501` | Prefix and extension remain                              |
| `+44 7700 900 123`          | `+44 [PHONE] 123`         | Prefix and last digits remain                            |
| Spaced digits `4 0 4...`    | `<PHONE_NUMBER>`          | Detected, but check whether the entire number was masked |

The key issue is that the scrubber sometimes recognizes only a portion of a phone number. A partially masked phone number can still reveal sensitive information.

### **D. Employee IDs**

Major detection gap

| Original information | Result                | Problem                              |
| -------------------- | --------------------- | ------------------------------------ |
| `EMP-04827`          | `[EMP_ID]`            | Successfully masked                  |
| `Staff #4082`        | Still present         | Missed                               |
| `RJP/2019/0451`      | Still present         | Missed                               |
| `A.M.-2021-EUR-007`  | Still present         | Missed                               |
| `Badge: CM18234`     | `<US_DRIVER_LICENSE>` | Detected, but classified incorrectly |

The scrubber recognizes the specific `EMP-` pattern, but it does not recognize the other employee ID formats.

### **E. Addresses**

Partially detected

The scrubber detects several location names and fragments of addresses. However, the original street addresses are not consistently removed.

For example:

- Tokyo address: the street details remain, and the postal code is partially affected.
- Dubai address: `Sheikh Zayed Road` is partly changed, but other address details remain.
- Mumbai address: parts of the road and locality remain.
- London address: `NW1 6XE` remains.

This means that detecting a city name is not the same as removing a complete postal address.

### **F. Financial information**

Partially protected

| Information           | Result           | Observation                                         |
| --------------------- | ---------------- | --------------------------------------------------- |
| `PRJ-2024-KHANJAR-EU` | Unchanged        | Correctly retained; this is a project code, not PII |
| IBAN                  | Partially masked | Only part of the IBAN was replaced                  |
| Masked card number    | Unchanged        | Not detected                                        |

The project code is an important negative test: a good scrubber should not remove information that is not PII.

### **G. Date**

`2024-03-15` was replaced with `<DATE_TIME>`.

This is a successful detection, although whether the date should be redacted depends on the document's privacy requirements.

## **2. Overall findings**

Your experiment demonstrates four important weaknesses:

1. **Partial masking:** The scrubber replaces only part of a sensitive value, leaving the rest visible.
2. **Missed formats:** Unusual email addresses, spelled-out phone numbers, and alternative employee ID formats are not detected.
3. **Incorrect classification:** Some information is detected under an unrelated category, such as `CM18234` being classified as a US driver's licence.
4. **Inconsistent detection:** Aisha Mohammed is masked in one chunk but not in another.

One important distinction: the 41 unique detections you counted are not 41 successfully protected PII values. They include partial matches, false positives, and repeated detections. We should not use that number as a success rate.

## **3. Next step: create the findings document**

We have enough evidence to document the results. In your Vocareum file browser, go to the `demos` folder and create these folders if they do not already exist:

```
demos/
└── docs/
    └── adr/
        └── pii_stress_test_findings.md
```

The file `pii_stress_test_findings.md` will record what your experiment actually found, rather than the expected results from the assignment.

**Before we create it, one important point:** The findings should distinguish between *detected*, *partially masked*, and *missed* information. That will make your report technically accurate.



In [33]:
from pathlib import Path

search_root = Path("/voc/work")

required_files = [
    "src/ingest/parsers.py",
    "src/ingest/chunkers.py",
    "src/ingest/pii.py",
]

for filename in required_files:
    matches = list(search_root.rglob(Path(filename).name))
    print(f"\n{filename}")

    if matches:
        for match in matches:
            print("FOUND:", match)
    else:
        print("NOT FOUND")


src/ingest/parsers.py
NOT FOUND

src/ingest/chunkers.py
NOT FOUND

src/ingest/pii.py
NOT FOUND
